# Quickstart and W&B

In [1]:
import os

import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

# If this import fails, run in a notebook cell: %pip install wandb
import wandb

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


## 1. W&B setup

In [ ]:
wandb.login()

project_name = "pytorch-fashion-mnist"
config = {
    "architecture": "NeuralNetwork",
    "dataset": "FashionMNIST",
    "epochs": 5,
    "batch_size": 64,
    "learning_rate": 1e-3,
    "optimizer": "SGD",
    "seed": 42,
}

if wandb.run is not None:
    wandb.finish()

run = wandb.init(
    project=project_name,
    name="quickstart-wandb",
    config=config,
)

torch.manual_seed(wandb.config.seed)

wandb.define_metric("trainer/global_step")
wandb.define_metric("train/*", step_metric="trainer/global_step")
wandb.define_metric("epoch")
wandb.define_metric("test/*", step_metric="epoch")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Mario\_netrc.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
wandb: Currently logged in as: zintom69 (zintomfromnowhere) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## 2. Datasets and Dataloaders

In [4]:
transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
])

training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=transform,
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=transform,
)

In [5]:
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

batch_size = wandb.config.batch_size

train_dataloader = DataLoader(training_data, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


## 3. Creating model

In [6]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits


model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


## 4. Optimizing

In [7]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=wandb.config.learning_rate)

wandb.watch(model, loss_fn, log="all", log_freq=100)

In [8]:
def train(dataloader, model, loss_fn, optimizer, epoch, log_interval=100):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.train()
    train_loss, correct = 0.0, 0

    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        pred = model(X)
        loss = loss_fn(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        batch_size = X.size(0)
        loss_value = loss.item()
        train_loss += loss_value * batch_size
        correct += (pred.argmax(1) == y).type(torch.float).sum().item()

        global_step = epoch * num_batches + batch + 1
        if batch % log_interval == 0:
            current = min((batch + 1) * batch_size, size)
            print(f"loss: {loss_value:>7f}  [{current:>5d}/{size:>5d}]")
            wandb.log(
                {
                    "trainer/global_step": global_step,
                    "train/loss": loss_value,
                    "train/learning_rate": optimizer.param_groups[0]["lr"],
                    "train/batch": batch,
                    "train/samples_seen": epoch * size + current,
                },
                step=global_step,
            )

    train_loss /= size
    train_accuracy = correct / size
    return train_loss, train_accuracy

In [9]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0.0, 0

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    test_accuracy = correct / size
    print(f"Test Error: \n Accuracy: {(100 * test_accuracy):>0.1f}%, Avg loss: {test_loss:>8f} \n")
    return test_loss, test_accuracy

## 5. Training models

In [10]:
epochs = wandb.config.epochs

for t in range(epochs):
    print(f"Epoch {t + 1}\n-------------------------------")
    train_loss, train_accuracy = train(train_dataloader, model, loss_fn, optimizer, epoch=t)
    test_loss, test_accuracy = test(test_dataloader, model, loss_fn)

    wandb.log(
        {
            "epoch": t + 1,
            "train/epoch_loss": train_loss,
            "train/epoch_accuracy": train_accuracy,
            "test/loss": test_loss,
            "test/accuracy": test_accuracy,
        },
        step=(t + 1) * len(train_dataloader),
    )

print("Done!")

Epoch 1
-------------------------------
loss: 2.301212  [   64/60000]
loss: 2.295208  [ 6464/60000]
loss: 2.283874  [12864/60000]
loss: 2.262271  [19264/60000]
loss: 2.255660  [25664/60000]
loss: 2.235273  [32064/60000]
loss: 2.226313  [38464/60000]
loss: 2.204105  [44864/60000]
loss: 2.202364  [51264/60000]
loss: 2.176423  [57664/60000]
Test Error: 
 Accuracy: 54.2%, Avg loss: 2.168638 

Epoch 2
-------------------------------
loss: 2.180879  [   64/60000]
loss: 2.141727  [ 6464/60000]
loss: 2.131764  [12864/60000]
loss: 2.109801  [19264/60000]
loss: 2.082061  [25664/60000]
loss: 2.064345  [32064/60000]
loss: 2.025877  [38464/60000]
loss: 2.029172  [44864/60000]
loss: 1.937385  [51264/60000]
loss: 1.946568  [57664/60000]
Test Error: 
 Accuracy: 60.7%, Avg loss: 1.920302 

Epoch 3
-------------------------------
loss: 1.920378  [   64/60000]
loss: 1.832684  [ 6464/60000]
loss: 1.873491  [12864/60000]
loss: 1.761599  [19264/60000]
loss: 1.755348  [25664/60000]
loss: 1.750082  [32064/600

## 6. Saving models

In [11]:
os.makedirs("ckpts", exist_ok=True)
checkpoint_path = "ckpts/quickstart_wandb_1.pth"

torch.save(model.state_dict(), checkpoint_path)

artifact = wandb.Artifact(
    name="quickstart-fashion-mnist-model",
    type="model",
    metadata=dict(wandb.config),
)
artifact.add_file(checkpoint_path)
run.log_artifact(artifact)

print(f"Saved PyTorch Model State to {checkpoint_path}")

Saved PyTorch Model State to ckpts/quickstart_wandb_1.pth


## 7. Loading models

In [12]:
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load(checkpoint_path, weights_only=True))

<All keys matched successfully>

In [13]:
model.eval()
x, y = test_data[0]

with torch.no_grad():
    x = x.unsqueeze(0).to(device)
    pred = model(x)
    predicted = classes[pred[0].argmax(0)]
    actual = classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')

wandb.log(
    {
        "example_prediction": wandb.Image(
            x.squeeze().cpu().numpy(),
            caption=f"Predicted: {predicted}, Actual: {actual}",
        )
    }
)

Predicted: "Ankle boot", Actual: "Ankle boot"


## Finish W&B run

In [14]:
wandb.finish()

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


epoch,▁▃▅▆█
test/accuracy,▁▆▅▇█
test/loss,█▆▄▂▁
train/batch,▁▂▃▃▄▆▆▇█▁▃▃▄▅▆▇█▁▃▃▅▆▆▇█▂▃▃▄▅▆▇█▁▂▃▄▅▆█
train/epoch_accuracy,▁▆▇▇█
train/epoch_loss,█▇▅▂▁
train/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/loss,██████▇▇▇▇▇▇▇▇▆▆▆▆▅▆▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁
train/samples_seen,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
trainer/global_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇████
epoch,5
